# Séries Temporais - Absenteísmo

In [ ]:
# =============================================================================
# Importações
# =============================================================================
import os
import logging
import pandas as pd
from typing import Tuple
from statsmodels.tsa.seasonal import seasonal_decompose
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from prophet import Prophet

# =============================================================================
# Configurações de Logging
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# =============================================================================
# Constantes
# =============================================================================
PATH_ARQUIVO = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
SHEET_NAME = "ATESTADOS"
EXCEL_ORIGIN = "1899-12-30"
OUTPUT_HTML = "Dashboard_Absenteismo_AFPESP.html"

# Cores corporativas
COR_AZUL = "#003366"
COR_LARANJA = "#FF6600"
COR_VERDE = "#2ca02c"
COR_VERMELHO = "#d62728"


# =============================================================================
# Funções
# =============================================================================

def carregar_dados(path: str, sheet: str) -> pd.DataFrame:
    logging.info("Carregando dados do Excel...")
    if not os.path.exists(path):
        logging.error(f"Arquivo não encontrado: {path}")
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

    df = pd.read_excel(path, sheet_name=sheet, engine="openpyxl")
    logging.info("Dados carregados com sucesso.")
    return df


def tratar_datas(df: pd.DataFrame) -> pd.DataFrame:
    logging.info("Tratando colunas de datas...")

    df["data_inicio"] = pd.to_datetime(df["data_inicio"], unit="D",
                                       origin=EXCEL_ORIGIN, errors="coerce")
    df["data_fim"] = pd.to_datetime(df["data_fim"], unit="D",
                                    origin=EXCEL_ORIGIN, errors="coerce")

    df = df.dropna(subset=["data_inicio", "data_fim"])
    return df


def calcular_severidade(df: pd.DataFrame) -> pd.DataFrame:
    logging.info("Calculando severidade (dias perdidos)...")

    df["dias_perdidos"] = (df["data_fim"] - df["data_inicio"]).dt.days + 1
    df["dias_perdidos"] = df["dias_perdidos"].clip(lower=1)

    return df


def agrupar_mensal(df: pd.DataFrame) -> pd.DataFrame:
    logging.info("Agrupando dados mensalmente...")

    df = df[df["data_inicio"].dt.year >= 2022]

    df = df.set_index("data_inicio")
    monthly = df.resample("MS").agg({
        "cod_empresa": "count",
        "dias_perdidos": "sum"
    })
    monthly = monthly.rename(columns={
        "cod_empresa": "Frequencia",
        "dias_perdidos": "Severidade"
    }).fillna(0)

    logging.info("Agrupamento mensal concluído.")
    return monthly


# 
# 🔮 PREVISÃO COM PROPHET
# 
def treinar_prophet(monthly: pd.DataFrame, periods: int = 12) -> Tuple[pd.DataFrame, Prophet]:
    """
    Treina modelo Prophet para previsão de frequência mensal.
    Retorna o forecast e o modelo treinado.
    """
    logging.info("Treinando modelo Prophet...")

    df_prophet = monthly.reset_index()[["data_inicio", "Frequencia"]]
    df_prophet.columns = ["ds", "y"]

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive"
    )
    model.fit(df_prophet)

    future = model.make_future_dataframe(periods=periods, freq="MS")
    forecast = model.predict(future)

    logging.info("Previsão Prophet concluída.")
    return forecast, model


def criar_dashboard(monthly: pd.DataFrame, decomp: seasonal_decompose,
                    forecast: pd.DataFrame) -> go.Figure:

    logging.info("Construindo dashboard com KPI Cards + Previsões Prophet...")

    # KPIs básicos
    total_atestados = int(monthly["Frequencia"].sum())
    media_mensal = float(monthly["Frequencia"].mean())
    total_dias_perdidos = int(monthly["Severidade"].sum())
    maior_pico = int(monthly["Frequencia"].max())
    mes_pico = monthly["Frequencia"].idxmax().strftime("%b/%Y")

    # 
    # Estrutura com 6 linhas (adicionamos Previsão Prophet)
    # 
    fig = make_subplots(
        rows=6, cols=1,
        row_heights=[0.18, 0.18, 0.18, 0.18, 0.18, 0.18],
        vertical_spacing=0.04,
        specs=[[{"type": "xy"}]] * 6,
        subplot_titles=(
            "KPI Executivos de Absenteísmo",
            "1. Série Original",
            "2. Tendência",
            "3. Sazonalidade",
            "4. Resíduos",
            "5. Previsão Prophet – Próximos 12 Meses"
        )
    )

    fig.update_xaxes(visible=False, row=1, col=1)
    fig.update_yaxes(visible=False, row=1, col=1)

    # 
    # KPI Cards usando shapes
    # 
    x_positions = [0.05, 0.30, 0.55, 0.80]
    kpis = [
        ("Total de Atestados", f"{total_atestados:,}".replace(",", "."), COR_AZUL),
        ("Média Mensal", f"{media_mensal:,.1f}".replace(",", "."), COR_LARANJA),
        ("Dias Perdidos", f"{total_dias_perdidos:,}".replace(",", "."), COR_VERDE),
        ("Maior Pico", f"{maior_pico} ({mes_pico})", COR_VERMELHO)
    ]

    for (titulo, valor, cor), x in zip(kpis, x_positions):
        fig.add_shape(
            type="rect",
            xref="paper", yref="paper",
            x0=x - 0.12, x1=x + 0.12,
            y0=0.88, y1=0.98,
            fillcolor=cor,
            opacity=0.15,
            line=dict(color=cor, width=2),
            row=1, col=1
        )
        fig.add_annotation(
            x=x, y=0.95, xref="paper", yref="paper",
            text=f"<b>{valor}</b>",
            showarrow=False,
            font=dict(size=26, color=cor),
            row=1, col=1
        )
        fig.add_annotation(
            x=x, y=0.89, xref="paper", yref="paper",
            text=titulo,
            showarrow=False,
            font=dict(size=14, color="black"),
            row=1, col=1
        )

    # 
    # 1. Série Original
    # 
    fig.add_trace(
        go.Scatter(
            x=monthly.index,
            y=monthly["Frequencia"],
            mode="lines+markers",
            name="Qtd Atestados",
            line=dict(color=COR_AZUL),
        ),
        row=2, col=1
    )

    # 
    # 2. Tendência
    # 
    fig.add_trace(
        go.Scatter(
            x=decomp.trend.index,
            y=decomp.trend,
            mode="lines",
            name="Tendência",
            line=dict(color=COR_LARANJA, width=3)
        ),
        row=3, col=1
    )

    # 
    # 3. Sazonalidade
    # 
    fig.add_trace(
        go.Scatter(
            x=decomp.seasonal.index,
            y=decomp.seasonal,
            mode="lines",
            name="Sazonalidade",
            line=dict(color=COR_VERDE)
        ),
        row=4, col=1
    )

    # 
    # 4. Resíduos
    # 
    fig.add_trace(
        go.Scatter(
            x=monthly.index,
            y=decomp.resid,
            mode="markers",
            name="Resíduos",
            marker=dict(color=COR_VERMELHO)
        ),
        row=5, col=1
    )
    fig.add_hline(y=0, line_dash="dot", row=5, col=1)

    # 
    # 5. PREVISÃO PROPHET
    # 
    forecast_plot = forecast.set_index("ds")

    fig.add_trace(
        go.Scatter(
            x=forecast_plot.index,
            y=forecast_plot["yhat"],
            mode="lines",
            name="Previsão (yhat)",
            line=dict(color=COR_AZUL, width=3)
        ),
        row=6, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=forecast_plot.index,
            y=forecast_plot["yhat_upper"],
            mode="lines",
            name="Limite Superior",
            line=dict(color=COR_VERDE, width=1, dash="dot")
        ),
        row=6, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=forecast_plot.index,
            y=forecast_plot["yhat_lower"],
            mode="lines",
            name="Limite Inferior",
            line=dict(color=COR_VERMELHO, width=1, dash="dot")
        ),
        row=6, col=1
    )

    fig.update_layout(
        title_text="Dashboard de Absenteísmo - Análise de Série Temporal + Previsão Prophet (AFPESP)",
        height=1500,
        template="plotly_white",
        hovermode="x unified",
        margin=dict(t=90)
    )

    return fig


def salvar_dashboard(fig: go.Figure, filename: str) -> None:
    fig.write_html(filename)
    logging.info(f"Dashboard salvo como: {filename}")


# =============================================================================
# Execução Principal
# =============================================================================
def main():
    logging.info("Iniciando processamento...")

    df = carregar_dados(PATH_ARQUIVO, SHEET_NAME)
    df = tratar_datas(df)
    df = calcular_severidade(df)

    monthly = agrupar_mensal(df)
    decomp = seasonal_decompose(monthly["Frequencia"], model="additive", period=12)

    forecast, model = treinar_prophet(monthly)

    fig = criar_dashboard(monthly, decomp, forecast)
    salvar_dashboard(fig, OUTPUT_HTML)

    logging.info("Processamento concluído com sucesso.")


if __name__ == "__main__":
    main()

2026-05-05 11:57:50,514 - INFO - Iniciando processamento...
2026-05-05 11:57:50,514 - INFO - Carregando dados do Excel...
2026-05-05 11:57:56,997 - INFO - Dados carregados com sucesso.
2026-05-05 11:57:56,997 - INFO - Tratando colunas de datas...
2026-05-05 11:57:57,035 - INFO - Calculando severidade (dias perdidos)...
2026-05-05 11:57:57,036 - INFO - Agrupando dados mensalmente...
2026-05-05 11:57:57,091 - INFO - Agrupamento mensal concluído.
2026-05-05 11:57:57,094 - INFO - Treinando modelo Prophet...
2026-05-05 11:57:57,711 - INFO - Chain [1] start processing
2026-05-05 11:57:58,583 - INFO - Chain [1] done processing
2026-05-05 11:57:58,626 - INFO - Previsão Prophet concluída.
2026-05-05 11:57:58,627 - INFO - Construindo dashboard com KPI Cards + Previsões Prophet...
2026-05-05 11:58:00,561 - INFO - Dashboard salvo como: Dashboard_Absenteismo_AFPESP.html
2026-05-05 11:58:00,564 - INFO - Processamento concluído com sucesso.
